# actBI Data Access Example

This notebook demonstrates how to access data using the shared.data library.

We'll explore:
- **Discovery**: Find assets, list partitions, search, and describe
- **Loading**: Get single assets, multiple partitions, or all data
- **Environments**: Switch between dev (S3), local, and prod

## Setup

Import libraries and set the default environment to **dev** (Supabase S3).

The `data` module is auto-imported by the actBI kernel.

In [3]:
import pandas as pd

from shared import data

# Set default environment to dev (Supabase S3)
data.use_env("dev")
print(f"Current environment: {data.current_env()}")

Current environment: dev


In [5]:
# List all available environments
print("Available environments:", data.list_environments())

Available environments: ['local', 'dev', 'prod']


## Discover Available Assets

Use `list_assets()` and `tree()` to explore what data is available.

In [6]:
# List all available assets
assets = data.list_assets()
print(f"Found {len(assets)} assets:\n")

# Group by layer
for layer in ["bronze", "silver", "gold"]:
    layer_assets = [a for a in assets if a.startswith(layer)]
    if layer_assets:
        print(f"{layer.upper()} ({len(layer_assets)}):")
        for asset in layer_assets:
            print(f"  - {asset}")
        print()

Found 0 assets:



In [7]:
# Show asset tree structure
data.tree()

{}

## Search and Describe Assets

Use `search()` to find assets by keyword and `describe()` to get detailed metadata.

In [8]:
# Search for assets related to "gdp"
gdp_assets = data.search("gdp")
print(f'Assets matching "gdp": {gdp_assets}')

# Get detailed metadata about an asset
info = data.describe("gold/economic/growth/gdp")
print(f"\nAsset: {info.asset_path}")
print(f"Partitioned: {info.is_partitioned}")
print(f"Partition count: {info.partition_count}")
print(f"Partitions: {info.partitions}")
print(f"Schema: {info.schema}")

Assets matching "gdp": []


NoCredentialsError: Unable to locate credentials

## Load GDP Data (Gold Layer)

Gold layer assets are partitioned by country. Use `list_partitions()` to see available countries,
then `get()` with a partition to load specific data.

In [ ]:
# Show available countries for GDP
countries = data.list_partitions("gold/economic/growth/gdp")
print(f"Available countries: {countries}")

# Load US GDP data
us_gdp = data.get("gold/economic/growth/gdp", partition="USA")
print(f"\nUS GDP records: {len(us_gdp)}")
print(f"Date range: {us_gdp['date'].min()} to {us_gdp['date'].max()}")
us_gdp.tail()

## Load Multiple Partitions

Use `get_all()` to load all partitions, or `get_many()` to load specific ones.

### get_all() - Load All Partitions

In [ ]:
# Load GDP for all countries (pd is already available)
all_gdp = data.get_all("gold/economic/growth/gdp")
print(f"Loaded {len(all_gdp)} countries: {list(all_gdp.keys())}")

# Combine into single DataFrame for comparison
gdp_combined = pd.concat([
    df.assign(country=country) for country, df in all_gdp.items()
])

# Show latest GDP by country
# TODO: Make sure our gold data is already indexed and presented properly.
latest_gdp = (
    gdp_combined
    .sort_values("date")
    .groupby("country")
    .last()[["date", "gdp", "source"]]
    .sort_values("gdp", ascending=False)
)
latest_gdp

### get_many() - Load Specific Partitions

In [ ]:
# Load only specific countries with get_many()
selected_gdp = data.get_many("gold/economic/growth/gdp", partitions=["USA", "CHN"])
print(f"Loaded {len(selected_gdp)} countries: {list(selected_gdp.keys())}")

for country, df in selected_gdp.items():
    print(f"  {country}: {len(df)} records")

## Visualize GDP Trends

Note: `matplotlib` is not auto-imported by the kernel - import it when needed.

In [ ]:
import matplotlib.pyplot as plt  # matplotlib not auto-imported

# Filter to recent data (2000+) for cleaner visualization
recent_gdp = gdp_combined.query('date >= "2000-01-01"')

fig, ax = plt.subplots(figsize=(12, 6))

for country in recent_gdp["country"].unique():
    country_data = recent_gdp.query(f'country == "{country}"')
    ax.plot(
        country_data["date"],
        country_data["gdp"] / 1e12,
        label=country,
        linewidth=2,
    )

ax.set_title("GDP Comparison Across Countries", fontsize=14)
ax.set_ylabel("GDP (Trillions)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Environment Switching

Use `data.env()` context manager to temporarily switch environments.
The **local** environment reads from `pipelines/_data/assets/` (checked into git).

In [ ]:
# Temporarily switch to local environment to load checked-in sample data
print(f"Current env: {data.current_env()}")

with data.env("local"):
    print(f"Inside context: {data.current_env()}")

    # Load FRED data from local assets (checked into git)
    local_assets = data.list_assets()
    print(f"Local assets available: {local_assets}")

    # Load a sample asset
    fred_data = data.get("bronze/fred/all_series")
    print(f"\nLoaded {len(fred_data)} FRED records from local")
    print(f"Series: {fred_data['series_id'].unique().tolist()}")

print(f"Back to: {data.current_env()}")

In [6]:
with data.env("local"):
    # Query financials with filters
    df = data.sec.financials(
        ticker="BRK.B",  # Single or list
        theme="profitability",  # Maps to XBRL concepts
        lazy=True,  # Set True for LazyFrame
        long=True,
    )

In [7]:
data.use_env("local")

In [8]:
data.get("silver/sec/xbrl_taxonomy")

,concept,label,documentation
0,AOCIAttributableToParentAbstract,AOCI Attributable to Parent [Abstract],
1,AOCIAttributableToParentNetOfTaxRollForward,"AOCI Attributable to Parent, Net of Tax [Roll ...",A roll forward is a reconciliation of a concep...
2,AOCIIncludingPortionAttributableToNoncontrolli...,AOCI Including Portion Attributable to Noncont...,
3,ASU201517TransitionAbstract,ASU 2015-17 Transition [Abstract] (Deprecated ...,
4,ASU201602TransitionAbstract,ASU 2016-02 Transition [Abstract],
...,...,...,...
17375,WriteOffOfDeferredDebtIssuanceCost,"Deferred Debt Issuance Cost, Writeoff",Write-off of amounts previously capitalized as...
17376,WrittenLoanCommitmentFairValueOptionMember,"Written Loan Commitment, Fair Value Option [Me...",This element represents a written loan commitm...
17377,YearEndAdjustmentMember,Year-End Adjustment [Member],A significant favorable or unfavorable adjustm...
17378,YearEndAdjustmentsEffectOfFourthQuarterEventsD...,"Effect of Fourth Quarter Events, Description",Description of a transaction recognized in the...


In [9]:
data.search("company")

['bronze/sec/company_facts',
 'bronze/sec/company_facts_download',
 'silver/sec/company_facts',
 'silver/sec/company_registry']

In [ ]:
with data.env("local"):
    data.get("silver/sec/company_registry").query('ticker.str.contains("BRK")')